In [ ]:
import os
import torch
import numpy as np
import pandas as pd

from dataset import AEDataModule
from model import AutoEncoder

import seaborn as sns
%matplotlib widget
import matplotlib.pyplot as plt

from views import GAIN_PIDS as gain_list

Set the theme for the plots

In [ ]:
# Set the theme for the plots 
sns.set_theme(context="notebook",
              style="whitegrid",
              palette="muted",
              font_scale=1.2,
             )

# Load the model and the dataset

In [ ]:
# Load the model and the dataset
v_num = 0
model_path = f"/home/IPP-HGW/orluca/devel/classificator/W7-X_QXT/AE360/version_{v_num}/best_model_.ckpt"
model = AutoEncoder.load_from_checkpoint(model_path)
model.eval() # Set the model to evaluation mode

data_dir = '../data'
filename = '20251028_all_data.npz'
data_module = AEDataModule(data_dir, filename, batch_size=-1, normalization_strategy='minmax')
data_module.setup()
# data_module.get_pids(gain_list)
# data_module.cut_time_intervals([(4,6)], '4to6_full.npz')
# data_module.setup()


In [ ]:
data_module.data['times']
data_module.data['profiles']


In [ ]:

device = torch.device('cpu')

full_dl = data_module.full_dataloader()

anomalous_pids = []
for batch in full_dl:
    x = batch['profile'].to(device) # The batch_size is 188058 and the num_diodes is 360
    y = model(x)
    # now i wish to compute the distance between x and y but for each single diode in each profile
    # the shape of x and y is (batch_size, num_diodes)
    # i want to compute the distance for each diode across the batch
    # so i want to compute the distance between x[:, i] and y[:, i]

    # # Get the distance for each diode and the overall distance for each profile
    # for i in range(x.shape[0]): # This is iterating over the batch dimension
    #     print(f"Profile {i}:")
    #     for j in range(x.shape[1]):
    #         distance = torch.norm(x[i, j] - y[i, j], p=2)
    #         print(f"Distance for diode {j}: {distance}")
    #     print("Overall distance for this profile: ", torch.norm(x[i] - y[i], p=2))
    #     print("Minimum distance for this profile: ", torch.min(x[i] - y[i]))
    #     print("Maximum distance for this profile: ", torch.max(x[i] - y[i]))
    #     print("Average distance for this profile: ", torch.mean(x[i] - y[i]))
    #     print("Median distance for this profile: ", torch.median(x[i] - y[i]))
    #     break

    # # Compute the pearson correlation coefficient for each profile across the batch
    for i in range(x.shape[0]):
        pearson_r = np.corrcoef(x[i].detach().numpy(), y[i].detach().numpy())[0, 1]
        if pearson_r < 0.9 and batch['pid'][i].item() not in anomalous_pids:
            anomalous_pids.append(batch['pid'][i].item())
            print(f"Profile {i} with PID {batch['pid'][i].item()} has a Pearson r of {pearson_r:.4f}")
            print(f"The time of this profile is {batch['time'][i].item()} seconds")


In [ ]:
# print out the anomalous pids
print("Anomalous PIDs: ", anomalous_pids)
print("Number of anomalous PIDs: ", len(anomalous_pids))

# Now I want to see if the anomalous PIDs are in the GAIN list
anomalous_gain_pids = [pid for pid in anomalous_pids if pid in gain_list]
print("Anomalous PIDs that are in the GAIN list: ", anomalous_gain_pids)
print("Number of anomalous PIDs that are in the GAIN list: ", len(anomalous_gain_pids))

print("Anomalous PIDs that are not in the GAIN list: ", [pid for pid in anomalous_pids if pid not in gain_list])
print("Number of anomalous PIDs that are not in the GAIN list: ", len([pid for pid in anomalous_pids if pid not in gain_list]))

print("Number of PIDs in the GAIN list: ", len(gain_list))
print("Number of PIDs in the dataset: ", len(data_module.full_data.pids))
print("Number of PIDs in the GAIN list that are in the dataset: ", len([pid for pid in gain_list if pid in data_module.full_data.pids]))

In [ ]:
# Now if the distance for the profile is more than 10 times the median distance, flag it as an anomaly
anomalies = []
median_distance = torch.median(torch.norm(x - y, p=2, dim=1))
print("Median distance: ", median_distance*1000)
for i in range(x.shape[0]):
    distance = torch.norm(x[i] - y[i], p=2)
    if distance > 10 * median_distance:
        # print(f"Profile {i} is an anomaly with distance {distance}")
    # get the pid of the profile and add it to a list of anomalies
        pid = data_module.data['pids'][i]
        # print(f"PID of the anomaly: {pid}")
        if pid not in anomalies:
            print(f"Detected anomaly for PID {pid} with distance {distance} at time {data_module.data['times'][i]}")
            anomalies.append(pid)

print("Anomalous PIDs: ", anomalies)
print("Number of anomalous profiles: ", len(anomalies))

# Also plot the distribution of distances for all profiles
distances = torch.norm(x - y, p=2, dim=1).detach().numpy()
sns.histplot(distances, bins=30, kde=True)
plt.xlabel("Distance between original and reconstructed profiles")
plt.ylabel("Frequency")
plt.title("Distribution of distances for all profiles")
plt.show()
plt.close('all')
# Do the same plot but highlight the training and validation datasets to understand if we are actually seeing anomalies or just a different distribution of data


In [ ]:
# 1. Define a threshold (e.g., the 95th percentile of your distances)
threshold = np.percentile(distances, 95) # in the cumulative distribution, 95% of the data is below this value 

# 2. Find the indices of profiles that exceed the threshold
fault_indices = np.where(distances > threshold)[0]

# 3. Extract the actual faulty data and their specific error scores
faulty_profiles = x[fault_indices]
fault_scores = distances[fault_indices]

print(f"Detected {len(fault_indices)} potential faults.")

# print the pids of the faulty profiles
faulty_pids = data_module.data['pids'][fault_indices]
print("PIDs of detected faults: ", np.unique(faulty_pids))

# print the time instants of the faulty profiles for a specific pid
for pid in np.unique(faulty_pids):
    pid_indices = np.where(faulty_pids == pid)[0]
    print(f"PID {pid} has {len(pid_indices)} potential faults at time instants: ", data_module.data['times'][fault_indices][pid_indices])

In [ ]:
# Sort indices by distance in descending order
top_faults = fault_indices[np.argsort(fault_scores)[::-1]]

plt.figure(figsize=(12, 4))
for i, idx in enumerate(top_faults[:3]):  # Look at the top 3
    plt.subplot(1, 3, i+1)
    plt.plot(x[idx].detach().numpy(), label='Actual (Faulty)', color='red', alpha=0.6)
    plt.plot(y[idx].detach().numpy(), label='Reconstructed (Clean)', color='blue', linestyle='--')
    plt.title(f"Score: {distances[idx]:.2f}")
    plt.legend()
plt.tight_layout()
plt.show()
plt.close('all')

In [ ]:
from views import plot_profiles_byPID

count = 0
for pid in gain_list:
    # Look if the pid exists in the dataset
    if pid in data_module.data['pids']:
        print(f"PID {pid} found in dataset. Plotting profiles...")
    else:
        print(f"PID {pid} not found in dataset. Please choose another PID.")

    plot_profiles_byPID(model, full_dl, device, pid, num_samples=1)
    count += 1
    if count >= 10:  # Limit to the first 3 PIDs for demonstration
        break

    plt.close('all')

